# ProtoHedge Asian-Call Real-Data Panel

This notebook runs the corrected 10-ticker ProtoHedge panel for an Asian short-call liability. Every episode follows one fixed listed hedge-option contract for all 20 steps, and both learned policies observe running-average moneyness so the path-dependent liability state is Markov with respect to the supplied features.

The reported outcome is the premium-excluding **liability offset**, not economic profit and loss. Candidate ProtoHedge configurations are selected only from validation periods; the locked test period is evaluated once after selection. Paired circular-block intervals account for dependence among overlapping episodes.


In [ ]:

from pathlib import Path
import sys
import os
import shutil
import json

cwd = Path.cwd().resolve()
if (cwd / 'world_real_torch.py').exists():
    REPO_ROOT = cwd
elif (cwd.parent / 'world_real_torch.py').exists():
    REPO_ROOT = cwd.parent
else:
    raise RuntimeError(f'Could not locate repo root from {cwd}')

os.environ.setdefault('MPLCONFIGDIR', str((REPO_ROOT / '.matplotlib-cache').resolve()))
Path(os.environ['MPLCONFIGDIR']).mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from IPython.display import display, Markdown

REPO_PARENT = REPO_ROOT.parent
repo_parent_str = str(REPO_PARENT)
sys.path = [p for p in sys.path if Path(p or '.').resolve() != REPO_PARENT]
sys.path.insert(0, repo_parent_str)
for mod_name in list(sys.modules):
    if mod_name == 'deephedging' or mod_name.startswith('deephedging.'):
        del sys.modules[mod_name]

from deephedging.real_data_sweep_torch import run_real_data_sweep, MODEL_SELECTION_VERSION
from deephedging.payoff_state import (
    ASIAN_PAYOFF_FEATURE,
    ASIAN_PAYOFF_STATE_VERSION,
    model_features_for_liability,
)
from deephedging.outcome_metrics import (
    OUTCOME_DEFINITION_VERSION,
    paired_panel_circular_block_bootstrap,
)
from deephedging.real_data_analysis_torch import (
    evaluate_saved_artifact,
    evaluate_saved_baselines,
    list_saved_artifacts,
    build_regime_frame,
    summarize_by_regime,
    prototype_usage_table,
    prototype_usage_by_regime,
)
from deephedging.prototype_extraction_torch import extract_agent_feature_matrix
from deephedging.proto_analysis_torch import load_prototype_payload

pd.set_option('display.max_columns', 300)
pd.set_option('display.width', 220)
plt.rcParams['figure.dpi'] = 120

FINAL_RUN_ROOT = REPO_ROOT / '.deephedging_real_runs' / 'asian_scientific_smoke_v2'
FINAL_RUN_ROOT.mkdir(parents=True, exist_ok=True)
PAPER_ASSET_DIR = REPO_ROOT / 'paper' / '.tmp_asian_scientific_smoke_assets_v2'
PAPER_ASSET_DIR.mkdir(parents=True, exist_ok=True)
ASIAN_PANEL_SUBDIR = 'asian_real_data_panel_scientific_v2'
ASIAN_PANEL_ASSET_DIR = PAPER_ASSET_DIR / ASIAN_PANEL_SUBDIR
ASIAN_PANEL_ASSET_DIR.mkdir(parents=True, exist_ok=True)


def _unique_path(path: Path) -> Path:
    path = Path(path)
    if not path.exists():
        return path
    stem = path.stem
    suffix = path.suffix
    i = 2
    while True:
        cand = path.with_name(f"{stem}_{i}{suffix}")
        if not cand.exists():
            return cand
        i += 1


def save_current_figure(name, subdir=ASIAN_PANEL_SUBDIR, dpi=180):
    subdir_path = PAPER_ASSET_DIR / subdir
    subdir_path.mkdir(parents=True, exist_ok=True)
    path = _unique_path(subdir_path / f'{name}.png')
    plt.gcf().savefig(path, dpi=dpi, bbox_inches='tight')
    print('saved figure:', path)
    return path


def save_dataframe(df, name, subdir=f'{ASIAN_PANEL_SUBDIR}/tables', latex=True, index=False):
    subdir_path = PAPER_ASSET_DIR / subdir
    subdir_path.mkdir(parents=True, exist_ok=True)
    csv_path = subdir_path / f'{name}.csv'
    df.to_csv(csv_path, index=index)
    print('saved table:', csv_path)
    if latex:
        tex_path = subdir_path / f'{name}.tex'
        try:
            tex = df.to_latex(index=index, escape=False, float_format=lambda x: f"{x:.6f}" if isinstance(x, (float, np.floating)) else str(x))
            tex_path.write_text(tex)
            print('saved latex:', tex_path)
        except Exception as exc:
            print('latex export skipped:', exc)
    return csv_path


def maybe_copy(src, subdir=f'{ASIAN_PANEL_SUBDIR}/copied_assets'):
    src = Path(src)
    if not src.exists():
        print('missing asset:', src)
        return None
    dst_dir = PAPER_ASSET_DIR / subdir
    dst_dir.mkdir(parents=True, exist_ok=True)
    dst = _unique_path(dst_dir / src.name)
    shutil.copy2(src, dst)
    print('copied asset:', dst)
    return dst


def expanded_feature_cols(feature_names, world, input_dim):
    per_step = world.data.features.per_step
    n_inst = int(world.nInst)
    cols = []
    count = 0
    for name in sorted(feature_names or ['price', 'delta', 'time_left']):
        if name in per_step:
            arr = np.asarray(per_step[name])
            width = 1 if arr.ndim == 2 else int(arr.shape[-1])
        elif name in ['delta', 'action']:
            width = n_inst
        elif name in ['pnl', 'cost']:
            width = 1
        else:
            width = 1
        if width == 1:
            cols.append(name)
            count += 1
        else:
            for j in range(width):
                cols.append(f'{name}_{j}')
                count += 1
    if count != int(input_dim):
        cols = [f'feature_{i}' for i in range(int(input_dim))]
    return cols


def select_best_row(best_df, preferred_selection, fallback_selections=()):
    for selection in (preferred_selection, *fallback_selections):
        sub = best_df[best_df['selection'] == selection]
        if not sub.empty:
            return sub.iloc[0]
    raise ValueError(f'Could not find preferred selection {preferred_selection} in best_df')

print('repo root:', REPO_ROOT)
print('final run root:', FINAL_RUN_ROOT)
print('paper assets:', PAPER_ASSET_DIR)


## Controls

The defaults below are set for the **full Asian-call panel run**.


In [ ]:

RUN_PANEL_SWEEP = True


## Panel Setup

This section loads the processed 10-ticker panel and configures the Asian-call sweep.


In [ ]:

PANEL_DIR = REPO_ROOT / 'Data' / 'NEW_PANEL'
MANIFEST_PATH = PANEL_DIR / 'panel_manifest.csv'
assert MANIFEST_PATH.exists(), f'Missing panel manifest: {MANIFEST_PATH}'
manifest = pd.read_csv(MANIFEST_PATH)
manifest = manifest.sort_values('ticker').reset_index(drop=True)
assert manifest['split_version'].eq('chronological-prewindow-v1').all(), 'Panel must use fixed chronological splits.'
assert manifest['option_path_version'].eq('fixed-optionid-v1').all(), 'Panel must use contract-consistent option paths.'
display(manifest)

PANEL_OUTPUT_ROOT = FINAL_RUN_ROOT / 'panel_sweeps'
PANEL_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
PANEL_SUMMARY_DIR = PANEL_OUTPUT_ROOT / 'panel_summary'
PANEL_SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

PANEL_TICKERS = ['SPY']   # smoke-test one ticker
SEEDS = (1234,)
EPOCHS = 1
NORMALIZE_REAL_WORLD = True
HEDGE_MODE = 'step'
USE_POSITION_BOUNDS = True
SKIP_COMPLETED_TICKERS = False
FORCE_RERUN_TICKERS = set()
PANEL_COMPLETION_FILES = (
    'fixed_chronological_splits.npz',
    'fixed_episode_metadata.csv',
    'sweep_metrics.csv',
    'sweep_test_summary.csv',
    'paper_model_comparison.csv',
    'paper_best_models.csv',
    'validation_model_selection.csv',
    'validation_selected_models.csv',
    'paired_block_bootstrap.csv',
    'paired_test_liability_offsets.npz',
    'paired_test_liability_offsets_metadata.csv',
    'sweep_config.json',
)

LIABILITY_TYPE = 'asian_call'
ASIAN_AVERAGE_TYPE = 'arithmetic'
ASIAN_START_STEP = 0
ASIAN_END_STEP = None
POLICY_FEATURES = model_features_for_liability(LIABILITY_TYPE)
assert ASIAN_PAYOFF_FEATURE in POLICY_FEATURES, 'Asian policy must observe running-average moneyness.'
EXPECTED_PAYOFF_STATE_VERSION = ASIAN_PAYOFF_STATE_VERSION

TRADE_BOUNDS = {
    'lbnd_as': -1.0,
    'ubnd_as': 1.0,
    'lbnd_av': -1.0,
    'ubnd_av': 1.0,
}
POSITION_BOUNDS = {
    'lbnd_delta_s': -1.0,
    'ubnd_delta_s': 1.0,
    'lbnd_delta_v': -1.0,
    'ubnd_delta_v': 1.0,
}
TRAIN_SELECTION_CFG = {
    'selection_metric': 'val_loss',
    'selection_alpha_action_abs': 0.005,
    'selection_alpha_delta_abs': 0.010,
    'selection_alpha_bound_occupancy': 0.100,
    'selection_alpha_path_bound_touch': 0.0,
}
TRAIN_REG_CFG = {
    'action_penalty_weight': 0.001,
    'delta_penalty_weight': 0.002,
}
ROBUST_SCREEN_CFG = {
    'max_bound_occupancy': 0.50,
    'max_path_touch_rate': None,
}
BOOTSTRAP_REPETITIONS = 20
BOOTSTRAP_CONFIDENCE = 0.95
BOOTSTRAP_BLOCK_LENGTH = 20

PANEL_SWEEP_MODE = 'frontier_only'  # smoke-test minimum grid
if PANEL_SWEEP_MODE == 'full_paper_grid':
    PROTOTYPE_COUNTS = (10, 25, 50, 100)
    PROTOTYPE_SOURCES = ('spot_delta', 'vanilla')
    WEIGHTED_SIMILARITY_OPTIONS = (False, True)
    LEARN_DISTANCE_FEATURE_WEIGHTS_OPTIONS = (False,)
else:
    PROTOTYPE_COUNTS = (2,)
    PROTOTYPE_SOURCES = ('spot_delta',)
    WEIGHTED_SIMILARITY_OPTIONS = (False,)
    LEARN_DISTANCE_FEATURE_WEIGHTS_OPTIONS = (False,)

PANEL_BASELINE_LABELS = {
    'unhedged': 'Unhedged',
    'spot_delta': 'Spot-Delta',
    'spot_delta_band': 'Spot-Delta Band',
    'vanilla': 'Vanilla DH',
}
PANEL_PROTO_SELECTION_LABELS = {
    'best_screened_proto_mean': 'ProtoHedge (Validation-Mean)',
    'best_screened_proto_cvar05': 'ProtoHedge (Validation-Tail)',
}
PANEL_PAPER_LABEL_ORDER = [
    'Unhedged',
    'Spot-Delta',
    'Spot-Delta Band',
    'Vanilla DH',
    'ProtoHedge (Validation-Mean)',
    'ProtoHedge (Validation-Tail)',
]
DEEP_DIVE_TICKER = 'QQQ'
DEEP_DIVE_SEED = int(SEEDS[0])
DEEP_DIVE_SELECTION = 'best_screened_proto_cvar05'

if PANEL_TICKERS is not None:
    manifest = manifest[manifest['ticker'].isin(PANEL_TICKERS)].copy().reset_index(drop=True)

if DEEP_DIVE_TICKER not in manifest['ticker'].tolist():
    DEEP_DIVE_TICKER = str(manifest['ticker'].iloc[0])

print('panel tickers:', manifest['ticker'].tolist())
print('panel sweep mode:', PANEL_SWEEP_MODE)
print('liability type:', LIABILITY_TYPE)
print('asian average type:', ASIAN_AVERAGE_TYPE)
print('payoff-state version:', EXPECTED_PAYOFF_STATE_VERSION)
print('policy features:', POLICY_FEATURES)
print('skip completed tickers:', SKIP_COMPLETED_TICKERS)
print('panel output root:', PANEL_OUTPUT_ROOT)


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
show = manifest[['ticker', 'n_episodes', 'n_feature_rows']].copy()
axes[0].bar(show['ticker'], show['n_episodes'])
axes[0].set_title('Episodes per ticker')
axes[0].tick_params(axis='x', rotation=45)
axes[1].bar(show['ticker'], show['n_feature_rows'])
axes[1].set_title('Feature rows per ticker')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
save_current_figure('asian_panel_manifest_diagnostics', subdir=ASIAN_PANEL_SUBDIR)
plt.show()


## Run Full Asian Panel Sweep

This is the expensive cell. It trains every candidate on the chronological training period, selects checkpoints and configurations on validation only, and then evaluates only the frozen models on test. Outputs from earlier protocols cannot satisfy the completion gate and will not be reused.


In [ ]:
def ticker_output_dir(ticker):
    return PANEL_OUTPUT_ROOT / f'{ticker}_{PANEL_SWEEP_MODE}_{EPOCHS}_{LIABILITY_TYPE}_step_robust_chrono_contract_asianstate_scientific_v2'


def ticker_is_complete(ticker):
    out_dir = ticker_output_dir(ticker)
    if not all((out_dir / name).exists() for name in PANEL_COMPLETION_FILES):
        return False
    try:
        sweep_cfg = json.loads((out_dir / 'sweep_config.json').read_text())
    except (OSError, json.JSONDecodeError):
        return False
    temporal = sweep_cfg.get('temporal_split', {})
    return (
        sweep_cfg.get('checkpoint_stage') == 'complete'
        and sweep_cfg.get('outcome_definition') == OUTCOME_DEFINITION_VERSION
        and sweep_cfg.get('premium_included') is False
        and sweep_cfg.get('model_selection_version') == MODEL_SELECTION_VERSION
        and temporal.get('split_version') == 'chronological-prewindow-v1'
        and temporal.get('option_path_version') == 'fixed-optionid-v1'
        and sweep_cfg.get('payoff_state_version') == EXPECTED_PAYOFF_STATE_VERSION
        and set(sweep_cfg.get('model_features', [])) == set(POLICY_FEATURES)
    )


def load_existing_ticker_metrics(ticker):
    out_dir = ticker_output_dir(ticker)
    metrics_path = out_dir / 'sweep_metrics.csv'
    if metrics_path.exists():
        metrics = pd.read_csv(metrics_path)
        assert metrics['outcome_definition'].eq(OUTCOME_DEFINITION_VERSION).all()
        assert metrics['model_selection_version'].eq(MODEL_SELECTION_VERSION).all()
        assert metrics['payoff_state_version'].eq(EXPECTED_PAYOFF_STATE_VERSION).all()
        assert metrics['model_features'].map(lambda value: set(str(value).split('|')) == set(POLICY_FEATURES)).all()
        return metrics
    return pd.DataFrame()


def run_one_ticker(row):
    ticker = row['ticker']
    out_dir = ticker_output_dir(ticker)
    out_dir.mkdir(parents=True, exist_ok=True)
    if (
        SKIP_COMPLETED_TICKERS
        and ticker not in FORCE_RERUN_TICKERS
        and ticker_is_complete(ticker)
    ):
        print(f'===== {ticker}: found completed corrected sweep, skipping retrain =====')
        return load_existing_ticker_metrics(ticker)

    print(f'===== {ticker}: running {PANEL_SWEEP_MODE} sweep for {LIABILITY_TYPE} =====')
    metrics_df = run_real_data_sweep(
        data_path=row['episode_path'],
        split_path=row['split_path'],
        episode_metadata_path=row['episode_metadata_path'],
        output_dir=out_dir,
        samples=30,
        seeds=SEEDS,
        epochs=EPOCHS,
        prototype_counts=PROTOTYPE_COUNTS,
        prototype_sources=PROTOTYPE_SOURCES,
        weighted_similarity_options=WEIGHTED_SIMILARITY_OPTIONS,
        learn_distance_feature_weights_options=LEARN_DISTANCE_FEATURE_WEIGHTS_OPTIONS,
        risk_measures=('cvar',),
        normalize=NORMALIZE_REAL_WORLD,
        hedge_mode=HEDGE_MODE,
        position_bounds=USE_POSITION_BOUNDS,
        trade_bounds=TRADE_BOUNDS,
        cumulative_bounds=POSITION_BOUNDS,
        liability_type=LIABILITY_TYPE,
        asian_average_type=ASIAN_AVERAGE_TYPE,
        asian_start_step=ASIAN_START_STEP,
        asian_end_step=ASIAN_END_STEP,
        selection_metric=TRAIN_SELECTION_CFG['selection_metric'],
        selection_alpha_action_abs=TRAIN_SELECTION_CFG['selection_alpha_action_abs'],
        selection_alpha_delta_abs=TRAIN_SELECTION_CFG['selection_alpha_delta_abs'],
        selection_alpha_bound_occupancy=TRAIN_SELECTION_CFG['selection_alpha_bound_occupancy'],
        selection_alpha_path_bound_touch=TRAIN_SELECTION_CFG['selection_alpha_path_bound_touch'],
        action_penalty_weight=TRAIN_REG_CFG['action_penalty_weight'],
        delta_penalty_weight=TRAIN_REG_CFG['delta_penalty_weight'],
        max_bound_occupancy=ROBUST_SCREEN_CFG['max_bound_occupancy'],
        max_path_touch_rate=ROBUST_SCREEN_CFG['max_path_touch_rate'],
        tuned_baseline_metric='liability_offset_mean',
        bootstrap_repetitions=BOOTSTRAP_REPETITIONS,
        bootstrap_confidence=BOOTSTRAP_CONFIDENCE,
        bootstrap_block_length=BOOTSTRAP_BLOCK_LENGTH,
    )
    return metrics_df


if RUN_PANEL_SWEEP:
    panel_metrics = {}
    for _, row in manifest.iterrows():
        panel_metrics[row['ticker']] = run_one_ticker(row)
else:
    print('RUN_PANEL_SWEEP=False, skipping sweep execution.')


## Aggregate Cross-Ticker Outputs

These cells collect the per-ticker sweep outputs and build the panel summary tables and figures.


In [ ]:
def load_panel_outputs_for_ticker(ticker):
    out_dir = ticker_output_dir(ticker)
    required = {
        'comparison': out_dir / 'paper_model_comparison.csv',
        'best': out_dir / 'paper_best_models.csv',
        'validation': out_dir / 'validation_model_selection.csv',
        'bootstrap': out_dir / 'paired_block_bootstrap.csv',
        'paired_offsets': out_dir / 'paired_test_liability_offsets.npz',
        'paired_metadata': out_dir / 'paired_test_liability_offsets_metadata.csv',
    }
    for label, path in required.items():
        assert path.exists(), f'Missing {label} output for {ticker}: {path}'
    comp = pd.read_csv(required['comparison'])
    best = pd.read_csv(required['best'])
    validation = pd.read_csv(required['validation'])
    bootstrap = pd.read_csv(required['bootstrap'])
    assert comp['outcome_definition'].eq(OUTCOME_DEFINITION_VERSION).all()
    assert best['selection_split'].eq('validation').all()
    assert best['model_selection_version'].eq(MODEL_SELECTION_VERSION).all()
    artifact_index = list_saved_artifacts(out_dir)
    return out_dir, comp, best, validation, bootstrap, artifact_index


panel_rows = []
panel_test_rows = []
panel_best_rows = []
panel_validation_rows = []
panel_bootstrap_rows = []
for _, mrow in manifest.iterrows():
    ticker = mrow['ticker']
    out_dir, comp, best, validation, bootstrap, artifact_index = load_panel_outputs_for_ticker(ticker)
    comp = comp.copy()
    comp['ticker'] = ticker
    comp['output_dir'] = str(out_dir)
    panel_test_rows.append(comp)
    validation = validation.copy()
    validation['ticker'] = ticker
    panel_validation_rows.append(validation)
    bootstrap = bootstrap.copy()
    bootstrap['ticker'] = ticker
    panel_bootstrap_rows.append(bootstrap)

    for model, label in PANEL_BASELINE_LABELS.items():
        sub = comp[comp['model'] == model]
        if sub.empty:
            continue
        row = sub.iloc[0].to_dict()
        row['paper_label'] = label
        row['selection'] = model
        panel_rows.append(row)

    for selection, label in PANEL_PROTO_SELECTION_LABELS.items():
        best_row = select_best_row(best, selection)
        assert best_row['selection_split'] == 'validation'
        row = best_row.to_dict()
        row['ticker'] = ticker
        row['paper_label'] = label
        row['output_dir'] = str(out_dir)
        panel_rows.append(row)
        panel_best_rows.append(row)

panel_df = pd.DataFrame(panel_rows)
panel_test_models_df = pd.concat(panel_test_rows, axis=0, ignore_index=True)
panel_validation_candidates_df = pd.concat(panel_validation_rows, axis=0, ignore_index=True)
panel_best_df = pd.DataFrame(panel_best_rows)
panel_bootstrap_df = pd.concat(panel_bootstrap_rows, axis=0, ignore_index=True)

panel_df['paper_label'] = pd.Categorical(
    panel_df['paper_label'], PANEL_PAPER_LABEL_ORDER, ordered=True
)
panel_df = panel_df.sort_values(['ticker', 'paper_label']).reset_index(drop=True)
panel_df.to_csv(PANEL_SUMMARY_DIR / 'panel_model_metrics.csv', index=False)
panel_test_models_df.to_csv(PANEL_SUMMARY_DIR / 'panel_test_models.csv', index=False)
panel_validation_candidates_df.to_csv(PANEL_SUMMARY_DIR / 'panel_validation_candidates.csv', index=False)
panel_best_df.to_csv(PANEL_SUMMARY_DIR / 'panel_validation_selected_rows.csv', index=False)
panel_bootstrap_df.to_csv(PANEL_SUMMARY_DIR / 'panel_per_ticker_block_bootstrap.csv', index=False)
print('saved corrected panel summaries to:', PANEL_SUMMARY_DIR)
display(panel_df[[
    'ticker', 'paper_label', 'liability_offset_mean_avg',
    'liability_offset_cvar05_avg', 'liability_offset_rmse_avg',
    'liability_offset_downside_deviation_avg',
    'pct_at_any_position_bound_avg', 'passes_validation_robust_screen',
]])


In [ ]:
def metric_heatmap(df, metric, title, fmt='.3f', cmap='viridis'):
    pivot = df.pivot(index='ticker', columns='paper_label', values=metric)
    pivot = pivot.reindex(columns=[c for c in PANEL_PAPER_LABEL_ORDER if c in pivot.columns])
    plt.figure(figsize=(12, max(4, 0.45 * len(pivot))))
    sns.heatmap(pivot, annot=True, fmt=fmt, cmap=cmap)
    plt.title(title)
    plt.tight_layout()
    return pivot


paper_subset = panel_df[panel_df['paper_label'].isin(PANEL_PAPER_LABEL_ORDER)].copy()
metric_heatmap(
    paper_subset,
    'liability_offset_mean_avg',
    'Asian-call mean liability offset by ticker and model',
)
save_current_figure('asian_panel_offset_mean_heatmap', subdir=ASIAN_PANEL_SUBDIR)
plt.show()

metric_heatmap(
    paper_subset,
    'liability_offset_cvar05_avg',
    'Asian-call 5% CVaR of liability offset by ticker and model',
    cmap='magma',
)
save_current_figure('asian_panel_offset_cvar_heatmap', subdir=ASIAN_PANEL_SUBDIR)
plt.show()

metric_heatmap(
    paper_subset,
    'liability_offset_rmse_avg',
    'Asian-call liability-offset RMSE by ticker and model',
    cmap='crest_r',
)
save_current_figure('asian_panel_offset_rmse_heatmap', subdir=ASIAN_PANEL_SUBDIR)
plt.show()

metric_heatmap(
    paper_subset,
    'pct_at_any_position_bound_avg',
    'Asian-call bound occupancy by ticker and model',
    fmt='.2f',
    cmap='rocket_r',
)
save_current_figure('asian_panel_bound_heatmap', subdir=ASIAN_PANEL_SUBDIR)
plt.show()

vanilla = paper_subset[paper_subset['paper_label'] == 'Vanilla DH'][[
    'ticker', 'liability_offset_mean_avg', 'liability_offset_cvar05_avg',
    'liability_offset_rmse_avg', 'pct_at_any_position_bound_avg',
]].rename(columns={
    'liability_offset_mean_avg': 'vanilla_mean',
    'liability_offset_cvar05_avg': 'vanilla_cvar',
    'liability_offset_rmse_avg': 'vanilla_rmse',
    'pct_at_any_position_bound_avg': 'vanilla_bound',
})
proto_labels = ['ProtoHedge (Validation-Mean)', 'ProtoHedge (Validation-Tail)']
proto_gap = paper_subset[paper_subset['paper_label'].isin(proto_labels)].merge(
    vanilla, on='ticker', how='left'
)
proto_gap['mean_difference_vs_vanilla'] = (
    proto_gap['liability_offset_mean_avg'] - proto_gap['vanilla_mean']
)
proto_gap['cvar_difference_vs_vanilla'] = (
    proto_gap['liability_offset_cvar05_avg'] - proto_gap['vanilla_cvar']
)
proto_gap['rmse_improvement_vs_vanilla'] = (
    proto_gap['vanilla_rmse'] - proto_gap['liability_offset_rmse_avg']
)
proto_gap['bound_reduction_vs_vanilla'] = (
    proto_gap['vanilla_bound'] - proto_gap['pct_at_any_position_bound_avg']
)
proto_gap.to_csv(PANEL_SUMMARY_DIR / 'panel_proto_vs_vanilla_differences.csv', index=False)

fig, axes = plt.subplots(4, 1, figsize=(12, 12), sharex=True)
for ax, col, title in [
    (axes[0], 'mean_difference_vs_vanilla', 'Mean-offset difference vs Deep Hedging'),
    (axes[1], 'cvar_difference_vs_vanilla', 'CVaR difference vs Deep Hedging'),
    (axes[2], 'rmse_improvement_vs_vanilla', 'RMSE improvement vs Deep Hedging'),
    (axes[3], 'bound_reduction_vs_vanilla', 'Bound-occupancy reduction vs Deep Hedging'),
]:
    for label, grp in proto_gap.groupby('paper_label', observed=True):
        ax.plot(grp['ticker'], grp[col], marker='o', label=label)
    ax.axhline(0.0, color='black', linewidth=1, linestyle='--')
    ax.set_title(title + ' (positive favors ProtoHedge)')
    ax.grid(True, alpha=0.3)
axes[0].legend()
plt.xticks(rotation=45)
plt.tight_layout()
save_current_figure('asian_panel_proto_vs_vanilla_differences', subdir=ASIAN_PANEL_SUBDIR)
plt.show()

# Equal-ticker panel inference: independently block-resample the chronological
# test episodes within each ticker, calculate paired differences, then average
# those ticker statistics with equal weight.
panel_interval_rows = []
for selection, paper_label in PANEL_PROTO_SELECTION_LABELS.items():
    candidate_by_ticker = {}
    benchmark_by_ticker = {}
    for ticker in manifest['ticker']:
        out_dir = ticker_output_dir(ticker)
        pair_meta = pd.read_csv(out_dir / 'paired_test_liability_offsets_metadata.csv')
        meta_row = pair_meta[pair_meta['selection'] == selection]
        assert len(meta_row) == 1, f'Expected one paired-offset row for {ticker} / {selection}'
        meta_row = meta_row.iloc[0]
        with np.load(out_dir / 'paired_test_liability_offsets.npz') as paired:
            candidate_by_ticker[ticker] = paired[meta_row['candidate_key']].copy()
            benchmark_by_ticker[ticker] = paired[meta_row['benchmark_key']].copy()
    rows = paired_panel_circular_block_bootstrap(
        candidate_by_ticker,
        benchmark_by_ticker,
        block_length=BOOTSTRAP_BLOCK_LENGTH,
        n_bootstrap=BOOTSTRAP_REPETITIONS,
        confidence=BOOTSTRAP_CONFIDENCE,
        random_state=314159 + len(panel_interval_rows),
    )
    for row in rows:
        panel_interval_rows.append({
            'selection': selection,
            'paper_label': paper_label,
            'selection_split': 'validation',
            'outcome_definition': OUTCOME_DEFINITION_VERSION,
            **row,
        })
panel_equal_ticker_bootstrap_df = pd.DataFrame(panel_interval_rows)
panel_equal_ticker_bootstrap_df.to_csv(
    PANEL_SUMMARY_DIR / 'panel_equal_ticker_block_bootstrap.csv', index=False
)
display(panel_equal_ticker_bootstrap_df)

# Per-ticker intervals show heterogeneity; panel intervals summarize the
# equal-ticker average used by the main result table.
forest_metrics = [
    'liability_offset_mean_difference',
    'liability_offset_cvar05_difference',
]
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
for ax, metric in zip(axes, forest_metrics):
    plot_df = panel_bootstrap_df[panel_bootstrap_df['metric'] == metric].copy()
    plot_df['label'] = plot_df['ticker'] + ' | ' + plot_df['selection'].map(PANEL_PROTO_SELECTION_LABELS)
    plot_df = plot_df.sort_values(['ticker', 'selection']).reset_index(drop=True)
    y = np.arange(len(plot_df))
    ax.errorbar(
        plot_df['estimate'], y,
        xerr=[plot_df['estimate'] - plot_df['ci_low'], plot_df['ci_high'] - plot_df['estimate']],
        fmt='o', capsize=3,
    )
    ax.axvline(0.0, color='black', linestyle='--', linewidth=1)
    ax.set_yticks(y, plot_df['label'])
    ax.set_title(metric.replace('_', ' '))
    ax.set_xlabel('positive favors ProtoHedge')
    ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
save_current_figure('asian_panel_paired_block_bootstrap_forest', subdir=ASIAN_PANEL_SUBDIR)
plt.show()


In [ ]:
panel_main = paper_subset[[
    'ticker', 'paper_label', 'liability_offset_mean_avg',
    'liability_offset_mean_std', 'liability_offset_cvar05_avg',
    'liability_offset_rmse_avg', 'liability_offset_downside_deviation_avg',
    'pct_at_any_position_bound_avg', 'passes_validation_robust_screen',
]].copy()
save_dataframe(panel_main, 'asian_panel_main_table', subdir=f'{ASIAN_PANEL_SUBDIR}/tables', latex=True, index=False)


def avg_table(df):
    rows = []
    for label, grp in df.groupby('paper_label', observed=True):
        rows.append({
            'paper_label': label,
            'mean_offset_avg': grp['liability_offset_mean_avg'].mean(),
            'mean_offset_std_across_tickers': grp['liability_offset_mean_avg'].std(ddof=0),
            'offset_cvar05_avg': grp['liability_offset_cvar05_avg'].mean(),
            'offset_rmse_avg': grp['liability_offset_rmse_avg'].mean(),
            'offset_downside_deviation_avg': grp['liability_offset_downside_deviation_avg'].mean(),
            'bound_occupancy_avg': grp['pct_at_any_position_bound_avg'].mean(),
            'n_tickers': grp['ticker'].nunique(),
        })
    out = pd.DataFrame(rows)
    out['paper_label'] = pd.Categorical(out['paper_label'], PANEL_PAPER_LABEL_ORDER, ordered=True)
    return out.sort_values('paper_label').reset_index(drop=True)


panel_average_table = avg_table(paper_subset)
save_dataframe(panel_average_table, 'asian_panel_average_table', subdir=f'{ASIAN_PANEL_SUBDIR}/tables', latex=True, index=False)


def rank_by_ticker(df, metric, ascending):
    ranked = []
    for ticker, grp in df.groupby('ticker'):
        sub = grp[['ticker', 'paper_label', metric]].copy()
        sub['rank'] = sub[metric].rank(ascending=ascending, method='min')
        sub['metric'] = metric
        ranked.append(sub)
    return pd.concat(ranked, ignore_index=True)


rank_df = pd.concat([
    rank_by_ticker(paper_subset, 'liability_offset_mean_avg', ascending=False),
    rank_by_ticker(paper_subset, 'liability_offset_cvar05_avg', ascending=False),
    rank_by_ticker(paper_subset, 'liability_offset_rmse_avg', ascending=True),
    rank_by_ticker(paper_subset, 'pct_at_any_position_bound_avg', ascending=True),
], ignore_index=True)
rank_summary = rank_df.groupby(['paper_label', 'metric'], observed=True)['rank'].mean().reset_index()
save_dataframe(rank_summary, 'asian_panel_average_rank_table', subdir=f'{ASIAN_PANEL_SUBDIR}/tables', latex=True, index=False)

winner_rows = []
for ticker, grp in paper_subset.groupby('ticker'):
    winner_rows.append({
        'ticker': ticker,
        'best_mean_offset_model': grp.sort_values('liability_offset_mean_avg', ascending=False).iloc[0]['paper_label'],
        'best_offset_cvar_model': grp.sort_values('liability_offset_cvar05_avg', ascending=False).iloc[0]['paper_label'],
        'lowest_offset_rmse_model': grp.sort_values('liability_offset_rmse_avg').iloc[0]['paper_label'],
        'lowest_bound_model': grp.sort_values('pct_at_any_position_bound_avg').iloc[0]['paper_label'],
    })
winner_table = pd.DataFrame(winner_rows).sort_values('ticker').reset_index(drop=True)
save_dataframe(winner_table, 'asian_panel_winner_table', subdir=f'{ASIAN_PANEL_SUBDIR}/tables', latex=True, index=False)

summary_rows = []
van = paper_subset[paper_subset['paper_label'] == 'Vanilla DH'].set_index('ticker')
for proto_label in ['ProtoHedge (Validation-Mean)', 'ProtoHedge (Validation-Tail)']:
    sub = paper_subset[paper_subset['paper_label'] == proto_label].set_index('ticker')
    common = sub.index.intersection(van.index)
    tmp = sub.loc[common].copy()
    summary_rows.append({
        'proto_label': proto_label,
        'n_tickers': len(common),
        'wins_mean_offset': int((tmp['liability_offset_mean_avg'] > van.loc[common, 'liability_offset_mean_avg']).sum()),
        'wins_offset_cvar': int((tmp['liability_offset_cvar05_avg'] > van.loc[common, 'liability_offset_cvar05_avg']).sum()),
        'wins_offset_rmse': int((tmp['liability_offset_rmse_avg'] < van.loc[common, 'liability_offset_rmse_avg']).sum()),
        'lower_bound_occupancy': int((tmp['pct_at_any_position_bound_avg'] < van.loc[common, 'pct_at_any_position_bound_avg']).sum()),
        'avg_mean_difference_vs_vanilla': float((tmp['liability_offset_mean_avg'] - van.loc[common, 'liability_offset_mean_avg']).mean()),
        'avg_cvar_difference_vs_vanilla': float((tmp['liability_offset_cvar05_avg'] - van.loc[common, 'liability_offset_cvar05_avg']).mean()),
        'avg_rmse_improvement_vs_vanilla': float((van.loc[common, 'liability_offset_rmse_avg'] - tmp['liability_offset_rmse_avg']).mean()),
        'avg_bound_reduction_vs_vanilla': float((van.loc[common, 'pct_at_any_position_bound_avg'] - tmp['pct_at_any_position_bound_avg']).mean()),
    })
proto_vs_vanilla_summary = pd.DataFrame(summary_rows)
save_dataframe(proto_vs_vanilla_summary, 'asian_panel_proto_vs_vanilla_summary', subdir=f'{ASIAN_PANEL_SUBDIR}/tables', latex=True, index=False)
save_dataframe(panel_equal_ticker_bootstrap_df, 'asian_panel_equal_ticker_block_bootstrap', subdir=f'{ASIAN_PANEL_SUBDIR}/tables', latex=True, index=False)

display(panel_average_table)
display(rank_summary)
display(winner_table)
display(proto_vs_vanilla_summary)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
proto_points = panel_validation_candidates_df.copy()
for ticker, grp in proto_points.groupby('ticker'):
    axes[0].scatter(
        grp['liability_offset_mean_avg'],
        grp['liability_offset_cvar05_avg'],
        alpha=0.35,
        label=ticker,
    )
axes[0].set_title('Asian-call validation mean vs CVaR across ProtoHedge candidates')
axes[0].set_xlabel('validation mean liability offset')
axes[0].set_ylabel('validation 5% CVaR of liability offset')
axes[0].grid(True, alpha=0.3)

for ticker, grp in proto_points.groupby('ticker'):
    axes[1].scatter(
        grp['validation_bound_occupancy_avg'],
        grp['liability_offset_mean_avg'],
        alpha=0.35,
        label=ticker,
    )
axes[1].set_title('Asian-call validation bound occupancy vs mean offset')
axes[1].set_xlabel('validation bound occupancy')
axes[1].set_ylabel('validation mean liability offset')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
save_current_figure('asian_panel_validation_candidate_scatter', subdir=ASIAN_PANEL_SUBDIR)
plt.show()


## Deep Dive: Representative Ticker

This section examines the Asian-call ProtoHedge policy frozen by the tail-oriented validation selector. Prototype concentration and regime use are computed on test only after model selection is complete.


In [ ]:

out_dir, comp, best, _validation, _bootstrap, artifact_index = load_panel_outputs_for_ticker(DEEP_DIVE_TICKER)
best_row = select_best_row(best, DEEP_DIVE_SELECTION, fallback_selections=('best_proto_cvar05', 'best_cvar05'))
TARGET_MODEL = str(best_row['model'])
print('deep dive ticker:', DEEP_DIVE_TICKER)
print('selection:', DEEP_DIVE_SELECTION)
print('target model:', TARGET_MODEL)

artifact_rows = artifact_index[(artifact_index['model_name'] == TARGET_MODEL) & (artifact_index['risk_measure'] == 'cvar')].copy()
if 'seed' in artifact_rows.columns and DEEP_DIVE_SEED in artifact_rows['seed'].tolist():
    artifact_dir = Path(artifact_rows[artifact_rows['seed'] == DEEP_DIVE_SEED].iloc[0]['artifact_dir'])
else:
    artifact_dir = Path(artifact_rows.iloc[0]['artifact_dir'])
print('artifact:', artifact_dir)

bundle, proto_result, proto_metrics = evaluate_saved_artifact(artifact_dir, split='test')
assert bundle['metadata']['payoff_state_version'] == EXPECTED_PAYOFF_STATE_VERSION
assert ASIAN_PAYOFF_FEATURE in bundle['metadata']['model_features']
print('validated policy state:', bundle['metadata']['model_features'])
baselines = evaluate_saved_baselines(artifact_dir, split='test')
vanilla_rows = artifact_index[(artifact_index['model_name'] == 'vanilla') & (artifact_index['risk_measure'] == 'cvar')].copy()
vanilla_artifact_dir = Path(vanilla_rows[vanilla_rows['seed'] == DEEP_DIVE_SEED].iloc[0]['artifact_dir']) if DEEP_DIVE_SEED in vanilla_rows['seed'].tolist() else Path(vanilla_rows.iloc[0]['artifact_dir'])
vanilla_bundle, vanilla_result, vanilla_metrics = evaluate_saved_artifact(vanilla_artifact_dir, split='test')

result_dict = {
    'proto': proto_result,
    'vanilla': vanilla_result,
    'spot_delta_band': baselines['spot_delta_band']['result'],
    'unhedged': baselines['unhedged']['result'],
}
regime_frame = build_regime_frame(bundle['eval_world'], result_dict)
regime_summary = pd.concat([
    summarize_by_regime(regime_frame, 'return_regime'),
    summarize_by_regime(regime_frame, 'vol_regime'),
    summarize_by_regime(regime_frame, 'drawdown_regime'),
], axis=0, ignore_index=True)
regime_summary.to_csv(PANEL_SUMMARY_DIR / f'{DEEP_DIVE_TICKER}_regime_summary.csv', index=False)
save_dataframe(regime_summary, f'{DEEP_DIVE_TICKER.lower()}_asian_regime_summary', subdir=f'{ASIAN_PANEL_SUBDIR}/tables', latex=False, index=False)
display(regime_summary.head())


In [ ]:
for regime_type in ['return_regime', 'vol_regime', 'drawdown_regime']:
    sub = regime_summary[regime_summary['regime_type'] == regime_type].copy()
    for metric in ['mean', 'cvar05', 'rmse', 'downside_deviation']:
        pivot = sub.pivot(index='regime', columns='series', values=metric)
        plt.figure(figsize=(8, 3))
        sns.heatmap(pivot, annot=True, fmt='.3f', cmap='viridis')
        plt.title(f'{DEEP_DIVE_TICKER} Asian-call: {metric} by {regime_type}')
        plt.tight_layout()
        save_current_figure(
            f'{DEEP_DIVE_TICKER.lower()}_asian_{regime_type}_{metric}',
            subdir=ASIAN_PANEL_SUBDIR,
        )
        plt.show()


In [ ]:

top_proto_df, full_proto_df = prototype_usage_table(bundle, proto_result, top_n=10)
assert ASIAN_PAYOFF_FEATURE in top_proto_df.columns, 'Prototype table is missing the Asian payoff state.'
usage_by_regime_df = prototype_usage_by_regime(proto_result, regime_frame)
save_dataframe(top_proto_df, f'{DEEP_DIVE_TICKER.lower()}_asian_prototype_top10', subdir=f'{ASIAN_PANEL_SUBDIR}/tables', latex=True, index=False)
save_dataframe(usage_by_regime_df, f'{DEEP_DIVE_TICKER.lower()}_asian_prototype_usage_by_regime', subdir=f'{ASIAN_PANEL_SUBDIR}/tables', latex=False, index=False)
display(top_proto_df)

plt.figure(figsize=(10, 4))
plot_df = top_proto_df.sort_values('usage_mean', ascending=False).head(10)
plt.bar(plot_df['prototype_index'].astype(str), plot_df['usage_mean'])
plt.title(f'{DEEP_DIVE_TICKER} Asian-call: top prototype usage')
plt.xlabel('prototype index')
plt.ylabel('mean weight')
plt.tight_layout()
save_current_figure(f'{DEEP_DIVE_TICKER.lower()}_asian_top_prototype_usage', subdir=ASIAN_PANEL_SUBDIR)
plt.show()

for regime_type in ['return_regime', 'vol_regime', 'drawdown_regime']:
    sub = usage_by_regime_df[usage_by_regime_df['regime_type'] == regime_type].copy()
    pivot = sub.pivot(index='regime', columns='prototype', values='mean_weight')
    plt.figure(figsize=(10, 3))
    sns.heatmap(pivot, annot=False, cmap='magma')
    plt.title(f'{DEEP_DIVE_TICKER} Asian-call: prototype usage by {regime_type}')
    plt.tight_layout()
    save_current_figure(f'{DEEP_DIVE_TICKER.lower()}_asian_{regime_type}_prototype_usage', subdir=ASIAN_PANEL_SUBDIR)
    plt.show()

payoff_state_cols = [
    'prototype_index', 'usage_mean', ASIAN_PAYOFF_FEATURE,
    'proto_action_0', 'proto_action_1',
]
payoff_state_proto_df = top_proto_df[payoff_state_cols].copy()
save_dataframe(
    payoff_state_proto_df,
    f'{DEEP_DIVE_TICKER.lower()}_asian_prototype_payoff_states',
    subdir=f'{ASIAN_PANEL_SUBDIR}/tables',
    latex=True,
    index=False,
)

state_plot = payoff_state_proto_df.sort_values('usage_mean', ascending=False)
fig, ax = plt.subplots(figsize=(9, 4))
usage_scale = state_plot['usage_mean'] / max(state_plot['usage_mean'].max(), 1e-12)
colors = plt.cm.viridis(usage_scale)
ax.bar(state_plot['prototype_index'].astype(str), state_plot[ASIAN_PAYOFF_FEATURE], color=colors)
ax.axhline(1.0, color='black', linestyle='--', linewidth=1, label='running average = strike')
ax.set_xlabel('prototype index, ordered by usage')
ax.set_ylabel('running-average moneyness')
ax.set_title(f'{DEEP_DIVE_TICKER} Asian-call: payoff state of dominant prototypes')
ax.legend(fontsize=8)
plt.tight_layout()
save_current_figure(f'{DEEP_DIVE_TICKER.lower()}_asian_prototype_payoff_state', subdir=ASIAN_PANEL_SUBDIR)
plt.show()



In [ ]:

usage_sorted = full_proto_df.sort_values('usage_mean', ascending=False).reset_index(drop=True).copy()
usage_sorted['cum_usage_mean'] = usage_sorted['usage_mean'].cumsum()
usage_sorted['k'] = np.arange(1, len(usage_sorted) + 1)
save_dataframe(usage_sorted[['k', 'prototype_index', 'usage_mean', 'cum_usage_mean']].head(50), f'{DEEP_DIVE_TICKER.lower()}_asian_prototype_concentration_table', subdir=f'{ASIAN_PANEL_SUBDIR}/tables', latex=True, index=False)

plt.figure(figsize=(8, 4))
plt.plot(usage_sorted['k'], usage_sorted['cum_usage_mean'], linewidth=2)
plt.axhline(0.8, color='black', linestyle='--', linewidth=1)
plt.xlabel('Top-k prototypes')
plt.ylabel('Cumulative usage mass')
plt.title(f'{DEEP_DIVE_TICKER} Asian-call: prototype concentration curve')
plt.grid(True, alpha=0.3)
plt.tight_layout()
save_current_figure(f'{DEEP_DIVE_TICKER.lower()}_asian_prototype_concentration', subdir=ASIAN_PANEL_SUBDIR)
plt.show()


In [ ]:

proto_path = bundle['metadata']['prototype_path']
payload = load_prototype_payload(proto_path)
prototypes_scaled = np.asarray(payload['prototypes'], dtype=np.float32)
scaler = payload['scaler']
feature_names = payload.get('feature_names', POLICY_FEATURES)
assert payload.get('payoff_state_version') == EXPECTED_PAYOFF_STATE_VERSION
assert ASIAN_PAYOFF_FEATURE in feature_names

x_raw, feature_names_sorted = extract_agent_feature_matrix(
    world=bundle['eval_world'],
    result=proto_result,
    feature_names=feature_names,
)
x_scaled = scaler.transform(x_raw)
feature_cols = expanded_feature_cols(feature_names_sorted, bundle['eval_world'], x_raw.shape[1])
assert ASIAN_PAYOFF_FEATURE in feature_cols

n_paths = int(bundle['eval_world'].data.market.hedges.shape[0])
n_steps = int(bundle['eval_world'].data.market.hedges.shape[1])

rows = []
for proto_idx in top_proto_df['prototype_index'].head(4).astype(int):
    dists = np.linalg.norm(x_scaled - prototypes_scaled[proto_idx], axis=1)
    nearest = np.argsort(dists)[:3]
    for rank, flat_idx in enumerate(nearest, start=1):
        path_idx = int(flat_idx // n_steps)
        step_idx = int(flat_idx % n_steps)
        row = {
            'prototype_index': int(proto_idx),
            'nearest_rank': int(rank),
            'distance_scaled': float(dists[flat_idx]),
            'path_index': path_idx,
            'step_index': step_idx,
        }
        for j, col in enumerate(feature_cols):
            row[col] = float(x_raw[flat_idx, j])
        rows.append(row)
nearest_proto_states_df = pd.DataFrame(rows).sort_values(['prototype_index', 'nearest_rank']).reset_index(drop=True)
save_dataframe(nearest_proto_states_df, f'{DEEP_DIVE_TICKER.lower()}_asian_nearest_prototype_states', subdir=f'{ASIAN_PANEL_SUBDIR}/tables', latex=True, index=False)
display(nearest_proto_states_df)

fig, axes = plt.subplots(min(4, len(nearest_proto_states_df['prototype_index'].unique())), 1, figsize=(10, 3.0 * min(4, len(nearest_proto_states_df['prototype_index'].unique()))), sharex=True)
if not isinstance(axes, np.ndarray):
    axes = np.array([axes])
for ax, proto_idx in zip(axes, nearest_proto_states_df['prototype_index'].drop_duplicates().tolist()[:4]):
    row = nearest_proto_states_df[nearest_proto_states_df['prototype_index'] == proto_idx].iloc[0]
    path_idx = int(row['path_index'])
    step_idx = int(row['step_index'])
    spot = np.asarray(bundle['eval_world'].data.features.per_step['spot'])[path_idx]
    running_avg = np.asarray(bundle['eval_world'].data.features.per_step['asian_running_average'])[path_idx]
    strike = float(np.asarray(bundle['eval_world'].data.features.per_path['strike'])[path_idx, 0])
    ax.plot(np.arange(len(spot)), spot, label=f'spot path {path_idx}')
    ax.plot(np.arange(len(running_avg)), running_avg, label='running average', linestyle='--')
    ax.axhline(strike, color='gray', linestyle=':', linewidth=1, label='strike')
    ax.axvline(step_idx, color='black', linestyle='--', linewidth=1)
    ax.scatter([step_idx], [running_avg[step_idx]], color='black', s=25, zorder=3)
    ax.set_title(f"prototype {proto_idx}: average/strike = {row[ASIAN_PAYOFF_FEATURE]:.3f}", fontsize=9)
    ax.set_ylabel('normalized level')
    ax.legend(fontsize=8)
axes[-1].set_xlabel('step')
plt.suptitle(f'{DEEP_DIVE_TICKER} Asian-call: nearest historical episodes for top prototypes', y=1.02)
plt.tight_layout()
save_current_figure(f'{DEEP_DIVE_TICKER.lower()}_asian_nearest_historical_episodes', subdir=ASIAN_PANEL_SUBDIR)
plt.show()


In [ ]:
# Copy corrected sweep assets into the unified paper asset directory.
for _, row in manifest.iterrows():
    out_dir = ticker_output_dir(row['ticker'])
    for filename in [
        'paper_model_comparison.csv',
        'paper_best_models.csv',
        'validation_model_selection.csv',
        'validation_selected_models.csv',
        'paired_block_bootstrap.csv',
        'paired_test_liability_offsets_metadata.csv',
    ]:
        maybe_copy(out_dir / filename, subdir=f'{ASIAN_PANEL_SUBDIR}/per_ticker_csv')
    for filename in [
        'sweep_test_liability_offset_mean.png',
        'sweep_test_liability_offset_cvar05.png',
        'sweep_test_liability_offset_rmse.png',
        'sweep_test_liability_offset_downside_deviation.png',
        'sweep_test_bound_saturation.png',
        'proto_validation_offset_vs_n_prototypes.png',
        'proto_validation_offset_cvar05_vs_n_prototypes.png',
    ]:
        maybe_copy(out_dir / filename, subdir=f'{ASIAN_PANEL_SUBDIR}/per_ticker_plots')

asset_manifest = pd.DataFrame(
    sorted([
        str(p.relative_to(PAPER_ASSET_DIR))
        for p in PAPER_ASSET_DIR.rglob('*') if p.is_file()
    ]),
    columns=['asset'],
)
save_dataframe(asset_manifest, 'asian_final_asset_manifest', subdir='.', latex=False, index=False)
display(asset_manifest.head(100))
